In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.messages.utils import trim_messages
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import base64
import filetype
from pypdf import PdfReader
from pathlib import Path

load_dotenv()

llm_nano = ChatOpenAI(
        model="openai/gpt-5.4-nano", # Твоя модель через прокси
        temperature=0,
        base_url="https://openai.api.proxyapi.ru/v1"
    )
llm_mini = ChatOpenAI(
        model="openai/gpt-5.4-mini", # Твоя модель через прокси
        temperature=0,
        base_url="https://openai.api.proxyapi.ru/v1"
    )

available_models = {
    "nano": llm_nano,
    "mini": llm_mini
}

def encode_image(image_path: str):
    with open(image_path, 'rb') as img:
        encoded_image = base64.b64encode(img.read()).decode("utf-8")
    return encoded_image

def get_mime_type(file_path: str):
    kind = filetype.guess(file_path)
    if kind is not None:
        return kind.mime
    return None

def extract_text_from_file(path: str) -> str:
    suffix = Path(path).suffix
    if suffix == ".pdf":
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
    else:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
    return text

# 1. Используем TypedDict для состояния
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

trimmer_by_count = trim_messages(
    max_tokens=4,           # Сохранит последние 5 сообщений
    strategy="last",
    token_counter=len,      # <-- Считаем каждое сообщение за 1 "токен"
    include_system=True,
)
    
# 2. Узел бота (чистая функция)
def chatbot(state: ChatState, config: RunnableConfig) -> ChatState:
    system_prompt = SystemMessage(
        content=(
            "Ты полезный AI-ассистент. "
            "Отвечай дружелюбно, понятно и по делу. "
            "Используй историю диалога при ответе."
        )
    )
    model_name = config.get("configurable", {}).get("model", "nano")
    llm = available_models[model_name]
    messages = [system_prompt] + trimmer_by_count.invoke(state["messages"])
    response = llm.invoke(messages)
    return {"messages": [response]}


builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

current_model = "nano"
config = {"configurable": {"thread_id": "session_1", "model": current_model}}

print("Чат-бот запущен! (Для выхода напиши 'выход' или 'exit')\n")

while True:
    user_text = input("Ты: ")
    
    if user_text.lower() in ["выход", "exit", "quit", "q"]:
        print("Бот: До встречи!")
        break

    match user_text.split(' ')[0] if user_text else "":
        case "/mini":
            current_model = "mini"
            print('Система переключена на mini')
            config = {"configurable": {"thread_id": "session_1", "model": current_model}}
            continue
        case "/nano":
            current_model = "nano"
            print('Система переключена на nano')
            config = {"configurable": {"thread_id": "session_1", "model": current_model}}
            continue
        case "/image":
            splitted_text = user_text.split(' ')
            path, raw_text = splitted_text[1], ' '.join(splitted_text[2:])
            encoded_image = encode_image(path)
            mime_type = get_mime_type(path).split('/')[1]
            image_data_url = f"data:image/{mime_type};base64,{encoded_image}"
            message = HumanMessage(content=[
                {"type": "text", "text": raw_text},
                {"type": "image_url", "image_url": {"url": image_data_url}}])
        case "/file":
            splitted_text = user_text.split(' ')
            path, raw_text = splitted_text[1], ' '.join(splitted_text[2:])
            file_text = extract_text_from_file(path)
            text_to_llm = "Ниже представлен файл в формате текста" + raw_text + "\n" + file_text
            message = HumanMessage(content=text_to_llm)
        case _:
            message = HumanMessage(content=user_text)

    input_state = {"messages": [message]}
    result_state = graph.invoke(input_state, config=config)
    bot_response = result_state["messages"][-1].content
    
    print(f"Бот: {bot_response}\n")

In [ ]:
def extract_text_from_file(path: str) -> str:
    suffix = Path(path).suffix
    if suffix == ".pdf":
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
    else:
        with open(path, "r", encoding="utf-8") as f:
            text = f.readlines()
    return ''.join(text)

# Асинхронный код с FastAPI

In [10]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.messages.utils import trim_messages
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import base64
import filetype
from pypdf import PdfReader
from pathlib import Path
from pydantic import BaseModel, Field
import uvicorn
from fastapi import FastAPI, HTTPException

load_dotenv()

SYSTEM_PROMPT = SystemMessage(
        content=(
            "Ты полезный AI-ассистент. "
            "Отвечай дружелюбно, понятно и по делу. "
            "Используй историю диалога при ответе."
        )
    )

llm_nano = ChatOpenAI(
        model="openai/gpt-5.4-nano", # Твоя модель через прокси
        temperature=0,
        base_url="https://openai.api.proxyapi.ru/v1"
    )
llm_mini = ChatOpenAI(
        model="openai/gpt-5.4-mini", # Твоя модель через прокси
        temperature=0,
        base_url="https://openai.api.proxyapi.ru/v1"
    )

available_models = {
    "nano": llm_nano,
    "mini": llm_mini
}

def encode_image(image_path: str):
    with open(image_path, 'rb') as img:
        encoded_image = base64.b64encode(img.read()).decode("utf-8")
    return encoded_image

def get_mime_type(file_path: str):
    kind = filetype.guess(file_path)
    if kind is not None:
        return kind.mime
    return None

def extract_text_from_file(path: str) -> str:
    suffix = Path(path).suffix
    if suffix == ".pdf":
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"
    else:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
    return text

# 1. Используем TypedDict для состояния
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

trimmer_by_count = trim_messages(
    max_tokens=4,           # Сохранит последние 4 сообщения
    strategy="last",
    token_counter=len,      # <-- Считаем каждое сообщение за 1 "токен"
    include_system=True,
)
    
# 2. Узел бота (чистая функция)
async def chatbot(state: ChatState, config: RunnableConfig) -> ChatState:
    system_prompt = SYSTEM_PROMPT
    model_name = config.get("configurable", {}).get("model", "nano")
    llm = available_models[model_name]
    messages = [system_prompt] + trimmer_by_count.invoke(state["messages"])
    response = await llm.ainvoke(messages)
    return {"messages": [response]}


builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)


In [ ]:
app = FastAPI(title="LangGraph Chat API")

# Схема входящего запроса
class ChatRequest(BaseModel):
    text: str = Field(
        description = "Обязательное поле: текст пользователя"
    )
    thread_id: str = Field(
        description = "Обязательное поле: ID диалога"
    )
    model: str = Field(default="nano",
        description = "Необязательное поле: по умолчанию nano, но пользователь может передать свою модель"
    )

# Эндпоинт для общения
@app.post("/chat")
async def chat_endpoint(request: ChatRequest):
    try:
        # 1. Формируем конфиг для графа
        config = {
            "configurable": {
                "thread_id": request.thread_id,
                "model": request.model
            }
        }

        user_text = request.text
        message = None

        match user_text.split(' ')[0] if user_text else "":
            case "/image":
                splitted_text = user_text.split(' ', 2)
                if len(splitted_text) < 2:
                    raise HTTPException(status_code=400, detail="Укажите путь к картинке: /image path [prompt]")
                path, raw_text = splitted_text[1], splitted_text[2]
                encoded_image = encode_image(path)
                mime_type = get_mime_type(path).split('/')[1]
                image_data_url = f"data:image/{mime_type};base64,{encoded_image}"
                message = HumanMessage(content=[
                    {"type": "text", "text": raw_text},
                    {"type": "image_url", "image_url": {"url": image_data_url}}])
            case "/file":
                splitted_text = user_text.split(' ')
                if len(splitted_text) < 2:
                    raise HTTPException(status_code=400, detail="Укажите путь к файлу: /file path [prompt]")
                path, raw_text = splitted_text[1], splitted_text[2]
                file_text = extract_text_from_file(path)
                text_to_llm = "Ниже представлен файл в формате текста" + raw_text + "\n" + file_text
                message = HumanMessage(content=text_to_llm)
            case _:
                message = HumanMessage(content=user_text)

        input_state = {"messages": [message]}
        result_state = await graph.ainvoke(input_state, config=config)

        # 4. Возвращаем последний ответ бота
        return {
            "status": "success",
            "response": result_state["messages"][-1].content,
            "thread_id": request.thread_id,
            "model_used": request.model
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [9]:
# Запуск сервера
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop